# Price impact and market depth

The OFI analysis established that mid-price changes are linear in order
flow imbalance, with a price-impact coefficient $\beta$ that differs across tickers and across
the trading day. This notebook tests the structural explanation proposed by

> Cont, R., Kukanov, A. and Stoikov, S. (2014). *The price impact of order book events.*
> Journal of Financial Econometrics, 12(1), 47-88.

namely that $\beta$ is inversely proportional to market depth:

$$\beta_{i,w} = \frac{c}{AD_{i,w}^{\lambda}}, \qquad \lambda \approx 1$$

where $AD_{i,w}$ is the average depth at the best quotes (mean of bid and ask size over the
window's events) for ticker $i$ in half-hour window $w$.

**Hypothesis.** Across 25 liquid tickers and 13 half-hour windows, $\log\beta$ is linear in
$\log AD$ with slope $-\lambda$, $\lambda$ significantly positive and close to 1. Falsified if
$\lambda \approx 0$, wrong-signed, or the log-log relation is visibly nonlinear.

**Method.** For each ticker, $\beta_{i,w}$ is fit per half-hour window from 10s intervals, keeping windows with at least
100 intervals and $t(\beta) > 2$. Then, following the paper, two regressions per ticker:

1. $\log\beta_{i,w} = a_i - \lambda_i \log AD_{i,w} + \epsilon$, so $\hat\lambda_i$ is minus
   the OLS slope.
2. $\beta_{i,w} = \alpha_i + c_i \, AD_{i,w}^{-\hat\lambda_i} + \epsilon$, giving $\hat c_i$
   conditional on $\hat\lambda_i$.

The window-level $\beta$ t-statistics use White standard errors and, as in the paper, the
per-ticker scaling regressions use Newey-West standard errors (the window series are
autocorrelated). Two panel estimates complement the per-ticker table, both with
standard errors clustered by ticker: a pooled fit (common $c$, identified by cross-sectional and
intraday depth variation together) and a fixed-effects fit (ticker-specific $c$, identified by
intraday variation only, which matches the paper's identification). The universe is the 25
most active tickers in the sample, chosen to span roughly four orders of magnitude in depth
(from GOOGL at tens of shares at the best quotes to SH and EEM at tens of thousands).

In [1]:
import numpy as np
import polars as pl
import statsmodels.api as sm
import plotly.graph_objects as go

from analysis.ofi import event_order_flow, iter_bbo, window_fits

TICKERS = [
    "SPY", "QQQ", "AMD", "INTC", "GOOGL",
    "TQQQ", "IWM", "DIA", "AAPL", "XLK",
    "MSFT", "SH", "MU", "EWZ", "XLI",
    "TLT", "XLV", "UVXY", "XLE", "SQQQ",
    "FB", "VXX", "GLD", "CSCO", "EEM",
]

wf = {t: window_fits(event_order_flow(bbo)) for t, bbo in iter_bbo(TICKERS)}

## Estimation sample

Stack the per-ticker window fits and drop windows that cannot support a reliable $\beta$.

In [2]:
panel = pl.concat([w.with_columns(pl.lit(t).alias("ticker")) for t, w in wf.items()])
kept = panel.filter((pl.col("n_intervals") >= 100) & (pl.col("t_beta") > 2))
print(f"{len(kept)} of {len(panel)} ticker-window pairs kept "
      f"(filters: n_intervals >= 100, t(beta) > 2)")

322 of 325 ticker-window pairs kept (filters: n_intervals >= 100, t(beta) > 2)


## Per-ticker estimates

One row per ticker: $\hat\lambda$ with 95% confidence interval and the $R^2$ of the log-log
regression, then $\hat c$ with its interval and the $R^2$ of the levels regression.

In [3]:
def depth_scaling_fits(w: pl.DataFrame, **fit_kwargs):
    """Two-step fit of beta = c / AD^lambda: log-log OLS for lambda, then levels OLS for c."""
    log_beta, log_ad = np.log(w["beta"].to_numpy()), np.log(w["avg_depth"].to_numpy())
    res_lam = sm.OLS(log_beta, sm.add_constant(log_ad)).fit(**fit_kwargs)
    lam = -res_lam.params[1]
    res_c = sm.OLS(w["beta"].to_numpy(), sm.add_constant(w["avg_depth"].to_numpy() ** -lam)).fit(**fit_kwargs)
    return res_lam, lam, res_c


NW = {"cov_type": "HAC", "cov_kwds": {"maxlags": 1}}  # Newey-West, as in the paper

rows = []
for t in TICKERS:
    w = kept.filter(pl.col("ticker") == t)
    res_lam, lam, res_c = depth_scaling_fits(w, **NW)
    slope_lo, slope_hi = res_lam.conf_int()[1]
    c_lo, c_hi = res_c.conf_int()[1]
    rows.append({
        "ticker": t, "windows": len(w),
        "lambda": lam, "lambda_lo": -slope_hi, "lambda_hi": -slope_lo, "r2_lambda": res_lam.rsquared,
        "c": res_c.params[1], "c_lo": c_lo, "c_hi": c_hi, "r2_c": res_c.rsquared,
    })
results = pl.DataFrame(rows)
covers = results.filter((pl.col("lambda_lo") <= 1) & (pl.col("lambda_hi") >= 1))
print(f"lambda = 1 lies inside the 95% interval for {len(covers)} of {len(results)} tickers")
with pl.Config(tbl_rows=25):
    display(results)

lambda = 1 lies inside the 95% interval for 20 of 25 tickers


ticker,windows,lambda,lambda_lo,lambda_hi,r2_lambda,c,c_lo,c_hi,r2_c
str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""SPY""",13,1.101684,0.363761,1.839608,0.355286,0.423016,0.251056,0.594975,0.582166
"""QQQ""",13,0.820358,0.553912,1.086805,0.708598,0.064178,0.04386,0.084496,0.767118
"""AMD""",13,0.855687,0.5896,1.121775,0.81472,0.08761,0.058835,0.116386,0.804149
"""INTC""",12,0.917261,0.785568,1.048953,0.914336,0.14159,0.123629,0.159552,0.91863
"""GOOGL""",11,-0.866844,-2.014158,0.280469,0.09882,0.00073,0.000007,0.001454,0.250529
"""TQQQ""",13,1.027488,0.791574,1.263403,0.832357,0.280875,0.212351,0.349399,0.80454
"""IWM""",13,0.343723,0.063062,0.624384,0.128854,0.00262,0.000517,0.004724,0.173606
"""DIA""",13,0.614025,0.22609,1.00196,0.202567,0.023312,0.00577,0.040853,0.179113
"""AAPL""",13,1.273358,0.87796,1.668755,0.674557,1.373672,0.7638,1.983544,0.537967


## Restricted model: $\lambda = 1$

The paper notes the restricted model fits nearly as well as the free exponent, and it makes $c$
directly comparable: with $\lambda = 1$ fixed, $\hat c$ is in ticks for every ticker. The
paper's stylized order-book model implies $c = 1/2$ and its grand mean estimate is 0.45.

In [4]:
rows = []
for t in TICKERS:
    w = kept.filter(pl.col("ticker") == t)
    res = sm.OLS(w["beta"].to_numpy(), sm.add_constant(1 / w["avg_depth"].to_numpy())).fit(**NW)
    c_lo, c_hi = res.conf_int()[1]
    rows.append({"ticker": t, "c": res.params[1], "c_lo": c_lo, "c_hi": c_hi, "r2": res.rsquared})
restricted = pl.DataFrame(rows)
print(f"median c under lambda = 1: {restricted['c'].median():.2f} ticks "
      f"(paper grand mean 0.45, stylized model 0.5)")
with pl.Config(tbl_rows=25):
    display(restricted)

median c under lambda = 1: 0.26 ticks (paper grand mean 0.45, stylized model 0.5)


ticker,c,c_lo,c_hi,r2
str,f64,f64,f64,f64
"""SPY""",0.236383,0.136352,0.336415,0.57381
"""QQQ""",0.180629,0.127174,0.234085,0.779139
"""AMD""",0.219848,0.14956,0.290137,0.809765
"""INTC""",0.224501,0.194223,0.254779,0.915629
"""GOOGL""",-0.617646,-1.243088,0.007796,0.221214
"""TQQQ""",0.244539,0.184655,0.304423,0.804388
"""IWM""",0.06748,0.015703,0.119256,0.191695
"""DIA""",0.11863,0.026632,0.210628,0.17611
"""AAPL""",0.420786,0.243986,0.597586,0.548385


## Panel estimates

Two ways to combine the kept ticker-windows, both with standard errors clustered by ticker. The
pooled regression imposes one common $c$, so cross-sectional and intraday depth variation both
identify $\lambda$. The fixed-effects regression demeans $\log\beta$ and $\log AD$ within each
ticker, allowing ticker-specific $c$ and using intraday variation only, which is the paper's
identification. Agreement between the two says the same scaling law holds across names as
within them.

In [5]:
groups = kept["ticker"].cast(pl.Categorical).to_physical().to_numpy()
CLUSTER = {"cov_type": "cluster", "cov_kwds": {"groups": groups}}

res_pooled, lam_pooled, _ = depth_scaling_fits(kept, **CLUSTER)
p_lo, p_hi = res_pooled.conf_int()[1]

within = kept.with_columns(
    (pl.col("beta").log() - pl.col("beta").log().mean().over("ticker")).alias("log_beta_dm"),
    (pl.col("avg_depth").log() - pl.col("avg_depth").log().mean().over("ticker")).alias("log_ad_dm"),
)
res_fe = sm.OLS(
    within["log_beta_dm"].to_numpy(), sm.add_constant(within["log_ad_dm"].to_numpy())
).fit(**CLUSTER)
f_lo, f_hi = res_fe.conf_int()[1]

panel_estimates = pl.DataFrame({
    "estimate": ["pooled lambda (common c)", "fixed-effects lambda (per-ticker c)", "median per-ticker lambda"],
    "value": [lam_pooled, -res_fe.params[1], results["lambda"].median()],
    "ci_lo": [-p_hi, -f_hi, None],
    "ci_hi": [-p_lo, -f_lo, None],
    "r2": [res_pooled.rsquared, res_fe.rsquared, None],
    "n": [len(kept), len(within), len(results)],
})
panel_estimates

estimate,value,ci_lo,ci_hi,r2,n
str,f64,f64,f64,f64,i64
"""pooled lambda (common c)""",0.998331,0.934641,1.062022,0.965712,322
"""fixed-effects lambda (per-tick…",0.914217,0.79497,1.033465,0.594227,322
"""median per-ticker lambda""",0.971955,null,null,null,25


## The scaling law
Every kept ticker-window as a point in $(AD, \beta)$ space, log-log axes. The line is the pooled
fit; a slope of exactly $-1$ would be the paper's $\lambda = 1$. The five tickers from the OFI
notebook keep their colors; the rest of the universe is gray.

In [6]:
COLOR = {"SPY": "#2a78d6", "QQQ": "#eb6834", "AMD": "#1baf7a", "INTC": "#eda100", "GOOGL": "#e87ba4"}

fig = go.Figure()
other = kept.filter(~pl.col("ticker").is_in(list(COLOR)))
fig.add_scatter(x=other["avg_depth"], y=other["beta"], mode="markers", name="Other (20)",
                marker=dict(color="#8c8c8c", size=6, opacity=0.5))
for t, c in COLOR.items():
    sub = kept.filter(pl.col("ticker") == t)
    fig.add_scatter(x=sub["avg_depth"], y=sub["beta"], mode="markers", name=t,
                    marker=dict(color=c, size=8))
ad = np.geomspace(kept["avg_depth"].min(), kept["avg_depth"].max(), 50)
a, b = res_pooled.params
fig.add_scatter(x=ad, y=np.exp(a) * ad**b, mode="lines", name=f"pooled fit (slope {b:.2f})",
                line=dict(color="#444444", width=2))
fig.update_xaxes(type="log", title_text="average depth at best quotes (shares)")
fig.update_yaxes(type="log", title_text="beta (ticks/share)")
fig.update_layout(height=550, width=950, template="plotly_white",
                  title=f"Price impact vs depth, {len(kept)} ticker-windows")
fig.show()

## Intraday seasonality

The paper's Figure 5: each ticker's $\beta$ and $AD$ normalized by its own day average, then
averaged across the 25 tickers per half-hour window. If impact is inversely proportional to
depth, the two curves should mirror each other around 1, with impact highest at the open where
the book is thinnest.

In [7]:
seasonal = (
    kept.with_columns(
        (pl.col("beta") / pl.col("beta").mean().over("ticker")).alias("beta_rel"),
        (pl.col("avg_depth") / pl.col("avg_depth").mean().over("ticker")).alias("depth_rel"),
    )
    .group_by("window")
    .agg(pl.col("beta_rel").mean(), pl.col("depth_rel").mean())
    .sort("window")
)
x = [w * 0.5 - 9.5 + 0.25 for w in seasonal["window"]]
fig = go.Figure()
fig.add_scatter(x=x, y=seasonal["beta_rel"], mode="lines+markers", name="beta / day average",
                line=dict(color="#2a78d6", width=2), marker=dict(size=8))
fig.add_scatter(x=x, y=seasonal["depth_rel"], mode="lines+markers", name="depth / day average",
                line=dict(color="#eb6834", width=2), marker=dict(size=8))
fig.add_hline(y=1, line=dict(color="#8c8c8c", width=1, dash="dot"))
fig.update_layout(height=380, width=950, template="plotly_white",
                  title="Intraday seasonality of price impact and depth, 25-ticker average",
                  xaxis_title="hours after 09:30", yaxis_title="ratio to ticker day average")
fig.show()

## Conclusion

The depth-scaling law is supported, with the exponent statistically indistinguishable from the
paper's $\lambda = 1$ under every estimator.

- The pooled fit (common $c$) gives $\hat\lambda$ = 0.998 with ticker-clustered 95% CI
  (0.935, 1.062) and $R^2$ = 0.966 over 322 ticker-windows.
- The fixed-effects fit, which matches the paper's within-ticker identification and allows
  ticker-specific $c$, gives $\hat\lambda$ = 0.914 with clustered CI (0.795, 1.033) and within
  $R^2$ = 0.594. Intraday depth variation alone recovers the law, just less precisely with 13
  windows per ticker.
- Per ticker, the median $\hat\lambda$ is 0.97 and the Newey-West 95% interval contains 1 for
  20 of 25 tickers (the paper reports 35 of 50). All 25 $\hat c$ estimates are positive, as the
  model requires.
- Agreement between the pooled and fixed-effects estimates says the same scaling holds across
  names as within them: the four orders of magnitude of cross-sectional depth in the figure lie
  on the same slope as each ticker's intraday variation.
- Imposing $\lambda = 1$ gives a median $\hat c$ of 0.26 ticks, the same order as the paper's
  grand mean of 0.45 and the stylized model's 1/2. As in the paper, $\hat c$ below 1/2 says
  mid-prices are somewhat more resilient than the average best-quote depth alone would suggest.
- The seasonality figure shows the mechanism directly: averaged across tickers, $\beta$ opens
  at 1.43 times its day average while depth opens at 0.79, and the close reverses both (0.68
  and 1.49). The curves mirror each other around 1, including a mid-afternoon depth dip on this
  particular day.
- The weak fits are visible in the table: IWM and DIA have intervals whose lower bounds cross
  zero, and GOOGL's own $\hat\lambda$ is negative with a very wide interval (fewest usable windows,
  narrow intraday depth range, $\beta$ dominated by the post-open collapse seen in the OFI
  notebook). This mirrors the paper, which flags its wide-spread, low-depth names (APOL, AZO,
  CME) as the bad fits.

Caveats: a single trading day against the paper's 21; 13 windows per ticker; $AD$ is
event-weighted, not time-weighted; $\hat c$ is estimated conditional on $\hat\lambda$ rather
than jointly.